### Stage 1 - offline Decision Transformer pretraining.
 
1. Roll out many behaviour policies (behaviors.py) in the TRAIN environment.
2. For each trajectory compute rho = mean reward per step (the conditioning value).
3. Train the transformer with supervised learning:
       (state, rho)  ->  action          (behaviour cloning, conditioned on rho)
       state         ->  future returns  (auxiliary "world-model" head)
4. Validate each epoch on the VAL split asking for a HIGH rho (default 90th pct),
   keep the best checkpoint.
 

In [1]:
import math
import os
import torch
import torch.nn.functional as F
import numpy as np

from trading_env import TradingEnv
from behaviours import make_random_behavior
import config as C
from model.model import DecisionTransformerPolicy,save_policy
from common import *


# Constants

In [2]:
episodes = 2000
epochs = 30
batch_size = 256
lr = 3e-4
wd = 0.01
d_model = 128
layers = 4
heads = 4
kv_heads = 2
dropout = 0.1
attn_window = None
aux_coef = 1.0
quantile = 0.9
news_train = None
news_val = None
seed = 0
bf16 = "store_true"
out = "checkpoints/dt"


In [3]:
from tqdm.auto import tqdm
def collect(env,n_episodes,seed):
    rng = np.random.default_rng(seed)
    env.reset(seed=seed)
    ts,ports,acts,lengths,ep_rho = [],[],[],[],[]
    for ep in tqdm(range(n_episodes)):
        obs,_ = env.reset()
        beh = make_random_behavior(env,rng)
        rews,done = [],False
        while not done:
            a = beh()
            ts.append(env.t)
            ports.append(obs["portfolio"])
            acts.append(a)
            obs,r,term,trunc,_= env.step(a)
            rews.append(r)
            done = term or trunc
        lengths.append(len(rews))
        ep_rho.append(float(np.sum(rews)/len(rews)))
    return (np.array(ts), np.array(ports, np.float32), np.array(acts, np.float32),
            np.array(lengths), np.array(ep_rho))
    

            


c:\Users\Iulian\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Main

In [4]:
torch.manual_seed(seed)
np.random.seed(seed)
device=  get_device()
os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
print('device',device)
news_tr, news_va = load_news(news_train), load_news(news_val)
env = TradingEnv(load_panel("train"), news=news_tr, episode_len=C.EPISODE_LEN)

val_env = TradingEnv(load_panel("val"), news=news_va)


device cuda


In [5]:
print(f"Collecting {episodes} behaviour episodes ...")
ts, ports, acts, lengths, ep_rho = collect(env, episodes,seed)


100%|██████████| 2000/2000 [00:37<00:00, 53.56it/s]


In [6]:
print("finite:", "rho", np.isfinite(ep_rho).all(), "| ports", np.isfinite(ports).all(),
      "| acts", np.isfinite(acts).all(), "| train feats", np.isfinite(env.feats).all(),
      "| val feats", np.isfinite(val_env.feats).all(), "| prices", np.isfinite(env.prices).all())


finite: rho True | ports True | acts True | train feats True | val feats True | prices True


In [7]:
rho_mean, rho_std = float(ep_rho.mean()), float(ep_rho.std())
qs = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
rho_q = {str(q): float(np.quantile(ep_rho, q)) for q in qs}


In [8]:
meta = {"rho_mean": rho_mean, "rho_std": rho_std,
            "rho_target": rho_q[str(quantile)] if str(quantile) in rho_q
            else float(np.quantile(ep_rho,quantile)),
            "rho_quantiles": rho_q, "horizons": list(C.AUX_HORIZONS)}
print(f"samples: {len(ts)} | rho mean {rho_mean:.3f} std {rho_std:.3f} | "
          f"quantiles {({k: round(v, 3) for k, v in rho_q.items()})}")
rho_z_target = (meta["rho_target"] - rho_mean) / (rho_std + 1e-9)


samples: 504000 | rho mean 0.051 std 0.167 | quantiles {'0.1': -0.164, '0.25': -0.04, '0.5': 0.056, '0.75': 0.144, '0.9': 0.256, '0.95': 0.337, '0.99': 0.484}


### Tensors

In [9]:
static = StaticTensors(env, device)
ts_t = torch.as_tensor(ts, dtype=torch.long, device=device)
port_t = torch.as_tensor(ports, device=device)
act_t = torch.as_tensor(acts, device=device).clamp(-0.98, 0.98)
rho_s = np.repeat(ep_rho, lengths)
rho_t = torch.as_tensor((rho_s - rho_mean) / (rho_std + 1e-9), dtype=torch.float32,device=device)



### Model

In [10]:
cfg = dict(n_assets=env.n, n_feat=env.nf, window=env.window,
               port_dim=env.observation_space["portfolio"].shape[0],
               act_dim=env.action_space.shape[0], n_aux_h=len(env.horizons),
               news_dim=env.news_dim if env.news is not None else 0,
               d_model=d_model, n_layers=layers, n_heads=heads,
               n_kv_heads=kv_heads, dropout=dropout,
               attn_window=attn_window)
model = DecisionTransformerPolicy(**cfg).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M | "
          f"seq_len {model.seq_len}")


params: 0.77M | seq_len 62


In [11]:
a,m = static.aux,static.aux_mask.bool()
scale = torch.stack([a[:,:,j][m[:,j].unsqueeze(1).expand(-1,env.n)].std() for j in range(a.shape[2])])
model.aux_scale.copy_(scale)


tensor([0.0203, 0.0428, 0.0847], device='cuda:0')

In [12]:
N = len(ts)
steps_per_epoch =  max(1,N // batch_size)
total = epochs * steps_per_epoch
warm = min(500 ,total//10+1)
opt = torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=wd,betas=(0.9,0.95))
scheduler = torch.optim.lr_scheduler.LambdaLR(
    opt,lambda s :min(1.0,(s+1)/warm) * max(0.05,0.5 * (1 + math.cos(math.pi * min(1.0,s/total))))
)
best = -1e9



In [13]:
for epoch in tqdm(range(1,epochs+1)):
    model.train()
    perm  =torch.randperm(N,device=device)
    bc_sum = aux_sum = 0.0
    for i in range(steps_per_epoch):
        idx = perm[i*  batch_size:(i+1) * batch_size]
        tb = ts_t[idx]
        obs = static.batch(tb)
        obs['portfolio'] = port_t[idx]
        with amp_ctx(device,bf16):
            mu,_,aux = model(obs,rho_t[idx])
        bc = F.mse_loss(mu,act_t[idx])
        target,mask = static.aux_batch(tb)
        al = aux_loss_fn(aux,target,mask,model.aux_scale)
        loss = bc + aux_coef * al
        if not torch.isfinite(loss):
            print("non-finite loss | bc", bc.item(), "aux", al.item(),
                    "| mu", torch.isfinite(mu).all().item(),
                    "| rho", torch.isfinite(rho_t[idx]).all().item(),
                    "| port", torch.isfinite(obs["portfolio"]).all().item(),
                    "| market", torch.isfinite(obs["market"]).all().item())
            raise SystemExit
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step()
        scheduler.step()
        bc_sum +=bc.item()
        aux_sum+=al.item()
    ev = evaluate_policy(model, val_env, rho_z_target, device, verbose=False)
    ag, spy = ev["metrics"]["Agent"], ev["metrics"].get("SPY buy&hold", {})
    score = ag["total_return"]
    print(f"epoch {epoch:3d} | bc {bc_sum / steps_per_epoch:.4f} | "
              f"aux {aux_sum / steps_per_epoch:.4f} | VAL agent {ag['total_return']:.1%} "
              f"(sharpe {ag['sharpe']:.2f}, dd {ag['max_dd']:.1%}) vs SPY "
              f"{spy.get('total_return', float('nan')):.1%}")
    if score > best:
            best = score
            save_policy(f"{out}_best.pt", model, meta)
    save_policy(f"{out}_last.pt", model, meta)

print(f"Done. Best val return {best:.1%}. Saved {args.out}_best.pt / _last.pt")


  3%|▎         | 1/30 [01:11<34:21, 71.08s/it]

epoch   1 | bc 0.3752 | aux 0.1558 | VAL agent 5.5% (sharpe 2.83, dd 0.7%) vs SPY 49.2%


  7%|▋         | 2/30 [02:20<32:49, 70.35s/it]

epoch   2 | bc 0.2333 | aux 0.0081 | VAL agent 6.5% (sharpe 1.84, dd 1.5%) vs SPY 49.2%


 10%|█         | 3/30 [03:31<31:44, 70.54s/it]

...

 97%|█████████▋| 29/30 [33:08<01:06, 66.56s/it]

epoch  29 | bc 0.1843 | aux 0.0014 | VAL agent 53.2% (sharpe 1.29, dd 15.7%) vs SPY 49.2%


100%|██████████| 30/30 [34:15<00:00, 68.53s/it]

epoch  30 | bc 0.1839 | aux 0.0014 | VAL agent 59.7% (sharpe 1.41, dd 14.8%) vs SPY 49.2%


NameError: name 'args' is not defined

In [14]:
print(f"Done. Best val return {best:.1%}. Saved {out}_best.pt / _last.pt")


Done. Best val return 391.3%. Saved checkpoints/dt_best.pt / _last.pt
